<a href="https://colab.research.google.com/github/sofiiamyshelova-prog/python_for_ds_tasks/blob/main/Copy_of_HW_%D0%9C%D0%B5%D1%82%D0%BE%D0%B4%D0%B8_%D0%BF%D0%BE%D0%BD%D0%B8%D0%B6%D0%B5%D0%BD%D0%BD%D1%8F_%D1%80%D0%BE%D0%B7%D0%BC%D1%96%D1%80%D0%BD%D0%BE%D1%81%D1%82%D1%96.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Домашнє завдання: Пониження розмірностей для Аналізу Портретів Клієнтів

#### Контекст
В цьому ДЗ ми попрацюємо з методами пониження розмірності на наборі даних для задачі аналізу портретів клієнтів (Customer Personality Analysis). **В попередньому ДЗ ми працювали з цими даними використовуючи кластеризацію, зараз використаємо кластеризацію і візуалізауємо результати з різними методами.**

Customer Personality Analysis - це аналіз різних сегментів клієнтів компанії. Цей аналіз дозволяє бізнесу краще розуміти своїх клієнтів і полегшує процес адаптації продуктів під конкретні потреби, поведінку та інтереси різних типів клієнтів.

Аналіз портретів клієнтів допомагає бізнесу змінювати свій продукт на основі цільової аудиторії, розділеної на різні сегменти. Наприклад, замість того, щоб витрачати гроші на маркетинг нового продукту для всіх клієнтів у базі даних компанії, бізнес може проаналізувати, який сегмент клієнтів найімовірніше придбає продукт, і потім зосередити маркетингові зусилля лише на цьому сегменті.

#### Вхідні дані
Вам надано набір даних з такими атрибутами:

**Характеристики користувачів:**
- `ID`: Унікальний ідентифікатор клієнта
- `Year_Birth`: Рік народження клієнта
- `Education`: Рівень освіти клієнта
- `Marital_Status`: Сімейний стан клієнта
- `Income`: Річний дохід домогосподарства клієнта
- `Kidhome`: Кількість дітей у домогосподарстві клієнта
- `Teenhome`: Кількість підлітків у домогосподарстві клієнта
- `Dt_Customer`: Дата реєстрації клієнта у компанії
- `Recency`: Кількість днів з моменту останньої покупки клієнта
- `Complain`: 1, якщо клієнт скаржився за останні 2 роки, 0 - якщо ні

**Продукти:**
- `MntWines`: Сума, витрачена на вино за останні 2 роки
- `MntFruits`: Сума, витрачена на фрукти за останні 2 роки
- `MntMeatProducts`: Сума, витрачена на м'ясні продукти за останні 2 роки
- `MntFishProducts`: Сума, витрачена на рибні продукти за останні 2 роки
- `MntSweetProducts`: Сума, витрачена на солодощі за останні 2 роки
- `MntGoldProds`: Сума, витрачена на золото за останні 2 роки

**Акції:**
- `NumDealsPurchases`: Кількість покупок, зроблених з використанням знижок
- `AcceptedCmp1`: 1, якщо клієнт прийняв пропозицію у першій кампанії, 0 - якщо ні
- `AcceptedCmp2`: 1, якщо клієнт прийняв пропозицію у другій кампанії, 0 - якщо ні
- `AcceptedCmp3`: 1, якщо клієнт прийняв пропозицію у третій кампанії, 0 - якщо ні
- `AcceptedCmp4`: 1, якщо клієнт прийняв пропозицію у четвертій кампанії, 0 - якщо ні
- `AcceptedCmp5`: 1, якщо клієнт прийняв пропозицію у п'ятій кампанії, 0 - якщо ні
- `Response`: 1, якщо клієнт прийняв пропозицію в останній кампанії, 0 - якщо ні

**Взаємодія з компанією:**
- `NumWebPurchases`: Кількість покупок, зроблених через вебсайт компанії
- `NumCatalogPurchases`: Кількість покупок, зроблених за каталогом
- `NumStorePurchases`: Кількість покупок, зроблених безпосередньо у магазинах
- `NumWebVisitsMonth`: Кількість відвідувань вебсайту компанії за останній місяць


Для початку, запустіть код нижче. Всі ці кроки ми робили в попередньому ДЗ і для того, щоб результати кластеризації у нас були схожими, потрібно аби передобробка була однаковою.

In [2]:
import pandas as pd

# 1. Завантаження даних
df = pd.read_csv('marketing_campaign.csv', sep='\t')

# 2. Обробка пропущених значень
df['Income_not_filled'] = df.Income.isna()
df.Income = df.Income.fillna(-1)

# 3. Обробка дати реєстрації
df.Dt_Customer = pd.to_datetime(df.Dt_Customer, format='%d-%m-%Y')
today = df.Dt_Customer.max()
df['days_lifetime'] = (today - df.Dt_Customer).dt.days
df['years_customer'] = df.Year_Birth.apply(lambda x: today.year - x)

# 4. Категоризація рівня освіти
df_education = pd.get_dummies(df.Education, prefix='education').astype(int)
df = pd.concat([df, df_education], axis=1)

# 5. Очищення сімейного стану
marital_status_map = {'Alone': 'Single', 'Absurd': 'Else', 'YOLO': 'Else'}
df['Marital_Status_clean'] = df.Marital_Status.map(marital_status_map)
df_ms = pd.get_dummies(df.Marital_Status_clean, prefix='marital').astype(int)
df = pd.concat([df, df_ms], axis=1)

# 6. Форматування доходу і видалення викиду
df.Income = df.Income.astype(int)
df = df[df.Income != 666666]

# 7. Створення фінального набору даних
X = df.drop(['ID', 'Dt_Customer', 'Education', 'Marital_Status', 'Marital_Status_clean'], axis=1)
X.reset_index(drop=True, inplace=True)

X

,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,Income_not_filled,days_lifetime,years_customer,education_2n Cycle,education_Basic,education_Graduation,education_Master,education_PhD,marital_Else,marital_Single
0,1957,58138,0,0,58,635,88,546,172,88,...,False,663,57,0,0,1,0,0,0,0
1,1954,46344,1,1,38,11,1,6,2,1,...,False,113,60,0,0,1,0,0,0,0
2,1965,71613,0,0,26,426,49,127,111,21,...,False,312,49,0,0,1,0,0,0,0
3,1984,26646,1,0,26,11,4,20,10,3,...,False,139,30,0,0,1,0,0,0,0
4,1981,58293,1,0,94,173,43,118,46,27,...,False,161,33,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2234,1967,61223,0,1,46,709,43,182,42,118,...,False,381,47,0,0,1,0,0,0,0
2235,1946,64014,2,1,56,406,0,30,0,0,...,False,19,68,0,0,0,0,1,0,0
2236,1981,56981,0,0,91,908,48,217,32,12,...,False,155,33,0,0,1,0,0,0,0
2237,1956,69245,0,1,8,428,30,214,80,30,...,False,156,58,0,0,0,1,0,0,0


### Завдання 1: Виконання кластеризації та пониження розмірності для візуалізації результатів

Ваше завдання — провести кластеризацію клієнтів та візуалізувати результати кластеризації, використовуючи метод головних компонент (PCA) для пониження розмірності даних.

#### Інструкції:

1. **Вибір ключових характеристик:**
   Давайте обмежимось тільки наступними хараткеристиками для кластеризації цього разу:
   - `Income`: Річний дохід домогосподарства клієнта
   - `Recency`: Кількість днів з моменту останньої покупки клієнта
   - `NumStorePurchases`: Кількість покупок, зроблених безпосередньо у магазинах
   - `NumDealsPurchases`: Кількість покупок, зроблених з використанням знижок
   - `days_lifetime`: Кількість днів з моменту реєстрації клієнта у компанії
   - `years_customer`: Вік клієнта
   - `NumWebVisitsMonth`: Кількість відвідувань вебсайту компанії за останній місяць
   Відберіть в наборі даних `X` лише ці характеристики.

2. **Стандартизація даних:**
   Використайте метод `StandardScaler` для стандартизації значень обраних характеристик.
   
   **Чому не MinMaxScaler:**
   - Для PCA краще використовувати StandardScaler, бо він вирівнює дисперсію ознак, на відміну від MinMaxScaler, що просто масштабує значення без врахування варіації.

   - Для K-Means також краще використовувати StandardScaler, бо алгоритм чутливий до масштабів: фічі з більшими значеннями сильніше впливають на обчислення відстаней, що може спотворити кластери.

3. **Кластеризація:**
   Проведіть кластеризацію клієнтів, використовуючи метод `KMeans` з трьома кластерами.

4. **Пониження розмірності:**
   Використайте метод головних компонент (PCA) для пониження розмірності даних до трьох компонент.

5. **Візуалізація результатів:**
   Використовуючи plolty express побудуйте 3D-графік розподілу клієнтів у просторі трьох головних компонент, де кольором позначено кластери.

6. **Опишіть, що спостерігаєте:**
   Чи кластеризація чітко розділила дані?

Далі ми детальніше проінтерпретуємо результати візуалізації і пониження розмірностей.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

silhouette = silhouette_score(X_scaled, clusters)

X_clastered=X
X_clastered['Cluster'] = clusters



In [4]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

In [5]:
import plotly.express as px

pca_df = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2', 'PC3']
)

pca_df['Cluster'] = clusters

# 3D-візуалізація
fig = px.scatter_3d(
    pca_df,
    x='PC1',
    y='PC2',
    z='PC3',
    color='Cluster',
    title='Customer Clusters in PCA 3D Space',
    labels={
        'PC1': 'Principal Component 1',
        'PC2': 'Principal Component 2',
        'PC3': 'Principal Component 3',
        'Cluster': 'Cluster'
    }
)

fig.show()

Кластеризація більш-менш чітко розділила дані

### Завдання 2: Аналіз результатів пониження розмірності

1. **Розрахунок частки поясненої дисперсії:**
   Визначте, яка частка загальної варіації даних пояснюється кожною з трьох головних компонент (PC1, PC2, PC3) за допомогою атрибуту `explained_variance_ratio_` об'єкта PCA. Виведіть результат на екран.

2. **Розрахунок кумулятивної частки поясненої дисперсії:**
   Обчисліть кумулятивну частку поясненої дисперсії для трьох головних компонент, щоб зрозуміти, скільки варіації даних пояснюється першими кількома компонентами.

In [6]:
print("Explained variance ratio:")

for i, ratio in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"PC{i}: {ratio:.4f}")

Explained variance ratio:
PC1: 0.2042
PC2: 0.0790
PC3: 0.0602


In [7]:
import numpy as np

cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

print("Cumulative explained variance:")

for i, ratio in enumerate(cumulative_variance, start=1):
    print(f"PC1-PC{i}: {ratio:.4f}")

Cumulative explained variance:
PC1-PC1: 0.2042
PC1-PC2: 0.2832
PC1-PC3: 0.3434


### Завдання 3: Інтерпретація "Loadings"

Продовжуємо інтерпретацію результатів `PCA`і познайомимось з новим поняттям `loadings`, яке допоможе нам знайти звʼязок між головними компонентами і оригінальними ознаками в наборі даних.

Ми зараз побудували візуалізацію кластерів точок даних в просторі трьох головних компонент. Але хочеться знайти звʼязок між головними компонентами і оригінальними ознаками. Для розуміння, які початкові характеристики даних мають найбільший вплив на ці головні компоненти, ми можемо використати атрибут `components_` методу `PCA`.

#### Що таке `pca.components_`?

`pca.components_` — це масив, який містить коефіцієнти (або "ваги"), що показують внесок кожної вихідної ознаки у кожну з головних компонент. Ці коефіцієнти ще називаються **"loading"** або "навантаженнями" компонент.

- **Loadings** (`навантаження`) відображають важливість кожної змінної (ознаки) для відповідної головної компоненти. Вони показують, яким чином змінні поєднуються, щоб утворити нові, зменшені вимірювання.
- Якщо коефіцієнт має високе абсолютне значення (як позитивне, так і негативне), це вказує на те, що відповідна змінна сильно впливає на головну компоненту.

#### Саме завдання
Ваше завдання — обчислити "навантаження" для кожної з головних компонент і інтерпретувати результати.

1. **Обчислення loadings для компонент:**
   Використайте атрибут `components_` об'єкта PCA для створення DataFrame, який відображатиме внесок кожної вихідної ознаки в кожну головну компоненту.

2. **Інтерпретація результатів:**
   Виведіть значення "навантажень" і проаналізуйте, які ознаки найбільше впливають на кожну головну компоненту.

In [8]:
# Loadings
loadings = pd.DataFrame(
    pca.components_,
    columns=X.drop(columns='Cluster').columns,
    index=['PC1', 'PC2', 'PC3']
)

print("PCA Loadings:")
print(loadings)

# Найважливіші ознаки для кожної компоненти
for pc in loadings.index:
    print(f"\nTop features for {pc}:")

    print(
        loadings.loc[pc]
        .sort_values(key=abs, ascending=False)
        .head(5)
    )

PCA Loadings:
     Year_Birth    Income   Kidhome  Teenhome   Recency  MntWines  MntFruits  \
PC1   -0.070400  0.324437 -0.252797 -0.045956  0.001427  0.303772   0.260020   
PC2    0.501282 -0.066676  0.095489 -0.426692 -0.020303 -0.106471   0.124823   
PC3    0.117148 -0.018870  0.082528 -0.036325 -0.111331  0.213711  -0.184516   

     MntMeatProducts  MntFishProducts  MntSweetProducts  ...  \
PC1         0.305601         0.269770          0.261746  ...   
PC2         0.112013         0.125256          0.119549  ...   
PC3        -0.035921        -0.188402         -0.159640  ...   

     Income_not_filled  days_lifetime  years_customer  education_2n Cycle  \
PC1          -0.021430       0.036916        0.070400           -0.015157   
PC2          -0.003405      -0.032213       -0.501282            0.087773   
PC3          -0.017309       0.128620       -0.117148           -0.031823   

     education_Basic  education_Graduation  education_Master  education_PhD  \
PC1        -0.066716

PC1 найбільше характеризується доходом та інтенсивністю покупок клієнта. Клієнти з високими значеннями PC1, як правило, мають вищий дохід і більше витрачають на товари та здійснюють більше покупок.

Year_Birth позитивно впливає на PC2

PC3 - реакція клієнта на маркетингові кампанії

###Завдання 4
Давайте проаналізуємо "навантаження" (**loadings**) для трьох головних компонент після вилучення ознаки `Income`. Це допоможе нам зрозуміти, як змінилася важливість інших ознак для кожної головної компоненти, коли одна з ключових характеристик (`Income`) була вилучена.

#### Кроки для проведення аналізу і ваше завдання:

1. Видаліть ознаку `Income` з нашого набору даних `X` і повторно виконайте PCA (метод головних компонент) для отримання нових "навантажень".

2. Обчисліть нові "навантаження" для трьох головних компонент на наборі даних без `Income`

3. Проаналізуйте, які ознаки мають найбільший вплив на кожну головну компоненту після вилучення `Income`.

4. Перегляньте, наскільки кожна з головних компонент пояснює дисперсію в даних без ознаки `Income`.

In [11]:
# Видаляємо Income
X_no_income = X.drop(columns=['Income','Cluster'])

# Масштабуємо дані
scaler_no_income = StandardScaler()
X_no_income_scaled = scaler_no_income.fit_transform(X_no_income)

# Виконуємо PCA
pca_no_income = PCA(n_components=3)
X_pca_no_income = pca_no_income.fit_transform(X_no_income_scaled)

loadings_no_income = pd.DataFrame(
    pca_no_income.components_,
    columns=X_no_income.columns,
    index=['PC1', 'PC2', 'PC3']
)

print("PCA Loadings without Income:")


for pc in loadings_no_income.index:
    print(f"\nTop features for {pc}:")
    print(
        loadings_no_income.loc[pc]
        .sort_values(key=abs, ascending=False)
        .head(5)
    )

PCA Loadings without Income:

Top features for PC1:
NumCatalogPurchases    0.329758
MntMeatProducts        0.321640
MntWines               0.317936
NumStorePurchases      0.293310
MntFishProducts        0.289156
Name: PC1, dtype: float64

Top features for PC2:
years_customer       0.505605
Year_Birth          -0.505605
Teenhome             0.424371
NumDealsPurchases    0.244278
education_PhD        0.189021
Name: PC2, dtype: float64

Top features for PC3:
Response        0.410475
AcceptedCmp4    0.357405
AcceptedCmp2    0.334486
AcceptedCmp5    0.306671
AcceptedCmp1    0.262271
Name: PC3, dtype: float64


In [12]:
explained_variance_no_income = pca_no_income.explained_variance_ratio_

for i, ratio in enumerate(explained_variance_no_income, start=1):
    print(f"PC{i}: {ratio * 100:.2f}%")


cumulative_variance_no_income = np.cumsum(
    pca_no_income.explained_variance_ratio_
)

for i, ratio in enumerate(cumulative_variance_no_income, start=1):
    print(f"PC1-PC{i}: {ratio * 100:.2f}%")

PC1: 18.96%
PC2: 8.11%
PC3: 6.20%
PC1-PC1: 18.96%
PC1-PC2: 27.07%
PC1-PC3: 33.28%


Тобто після видалення Income три компоненти пояснюють 33.28% дисперсії замість 34.34%. Це невелике зменшення, тому Income справді була важливою, але не єдина ознака, яка формує основну структуру даних.

вилучення Income не зруйнувало структуру PCA — інші ознаки, особливо показники покупок, змогли значною мірою зберегти цю інформацію.

### Завдання 5: Візуалізація кластеризації за допомогою t-SNE

Ваше завдання — використати метод t-SNE для візуалізації результатів кластеризації клієнтів у двовимірному просторі. Метод t-SNE допомагає знизити розмірність даних та зберегти локальні структури в даних, що робить його ефективним для візуалізації високорозмірних даних. Ми також зможемо порівняти результат цього методу з РСА.

1. Використайте метод t-SNE для зниження розмірності до 2х вимірів даних, які включають ознаки всі, що і в завданні 1, а також були відмасштабовані перед пониженням розмірностей.

2. Створіть новий DataFrame з координатами, отриманими після застосування t-SNE, та додайте до нього мітки кластерів.

3. Побудуйте інтерактивний 2D-графік розподілу клієнтів, де кольором буде позначено різні кластери і проаналізуйте графік з рекомендаціями нижче (можливо треба буде вивести додаткові візуалізації чи таблиці для інтерпретації, але треба прям зрозуміти, які ознаки формують який кластер і чим кластери відрізняються одне від одного).

  **Опишіть отримані кластери з точки зору ознак.**

4. Опишіть відмінність графіка tSNE від PCA.

#### ЯК можна інтерпретувати з t-SNE?

Хоча t-SNE не надає "компонентів" як РСА, він забезпечує низьковимірне представлення даних, яке можна візуально інтерпретувати:

- **Кластери:** t-SNE особливо добре показує кластери подібних точок. Якщо ви бачите чітко визначені кластери на графіку t-SNE, це свідчить про наявність груп схожих спостережень у ваших даних. Проаналізувати їх можемо, якщо додамо дані в `hover_data` або якщо якісь з даних виведемо як розмір чи форма точок на візуалізації. Також корисно може бути вивести середні значення ознак по кластерам.
- **Локальна структура:** Відносне розташування точок одного кластеру на графіку t-SNE може допомогти вам зрозуміти, які дані подібні між собою.
- **Глобальна структура:** Будьте обережні; t-SNE менш надійний для відображення глобальних структур (наприклад, відстаней між кластерами) у порівнянні з PCA, бо t-SNE націлений на збереження саме локальних структур.

In [15]:
from sklearn.manifold import TSNE

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30
)

X_tsne = tsne.fit_transform(X_scaled)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

tsne_df = pd.DataFrame(
    X_tsne,
    columns=['tSNE1', 'tSNE2']
)

tsne_df['Cluster'] = clusters

tsne_df.head()

,tSNE1,tSNE2,Cluster
0,12.029774,-3.809126,1
1,-29.511986,-18.387712,0
2,9.425376,-13.332931,2
3,-54.677212,4.742729,0
4,34.832348,39.250587,0


In [16]:
fig = px.scatter(
    tsne_df,
    x='tSNE1',
    y='tSNE2',
    color='Cluster',
    title='Customer Clusters — t-SNE',
    hover_data=['Cluster']
)

fig.show()

Отримана двовимірна візуалізація показує наявність локальних груп схожих клієнтів, однак три кластери не є повністю відокремленими та частково перекриваються.

Аналіз середніх значень ознак показав, що Cluster 0 характеризується низьким доходом та низькою купівельною активністю: клієнти мають низькі значення витрат практично за всіма категоріями товарів.

Cluster 1 є найбільш цінним з точки зору купівельної поведінки: клієнти мають високий дохід та дуже високі витрати на різні категорії товарів, хоча значне значення Recency свідчить про те, що вони могли давно не здійснювати останню покупку.

Cluster 2 характеризується старшими клієнтами, більшою тривалістю взаємодії з компанією та більшою кількістю підлітків у домогосподарстві. Їхній дохід близький до середнього, а купівельна активність є помірною.

